In [3]:
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd().parent  # one level up from Silver
INPUT_DIR = PROJECT / "Bronze"
OUTPUT_DIR = Path.cwd()  # Silver itself, since the script runs from here

# Maps each Finnish municipality (as it appears in Traficom data) to its region.
MUNICIPALITY_TO_REGION = {
    'Akaa': 'Pirkanmaa', 'Alajärvi': 'South Ostrobothnia', 'Alavieska': 'North Ostrobothnia',
    'Alavus': 'South Ostrobothnia', 'Asikkala': 'Päijät-Häme', 'Askola': 'Uusimaa',
    'Aura': 'Southwest Finland', 'Enonkoski': 'South Savo', 'Enontekiö': 'Lapland',
    'Espoo': 'Uusimaa', 'Eura': 'Satakunta', 'Eurajoki': 'Satakunta',
    'Evijärvi': 'South Ostrobothnia', 'Forssa': 'Kanta-Häme', 'Haapajärvi': 'North Ostrobothnia',
    'Haapavesi': 'North Ostrobothnia', 'Hailuoto': 'North Ostrobothnia', 'Halsua': 'Central Ostrobothnia',
    'Hamina': 'Kymenlaakso', 'Hankasalmi': 'Central Finland', 'Hanko': 'Uusimaa',
    'Harjavalta': 'Satakunta', 'Hartola': 'Päijät-Häme', 'Hattula': 'Kanta-Häme',
    'Hausjärvi': 'Kanta-Häme', 'Heinola': 'Päijät-Häme', 'Heinävesi': 'North Karelia',
    'Helsinki': 'Uusimaa', 'Hirvensalmi': 'South Savo', 'Hollola': 'Päijät-Häme',
    'Huittinen': 'Satakunta', 'Humppila': 'Kanta-Häme', 'Hyrynsalmi': 'Kainuu',
    'Hyvinkää': 'Uusimaa', 'Hämeenkyrö': 'Pirkanmaa', 'Hämeenlinna': 'Kanta-Häme',
    'Ii': 'North Ostrobothnia', 'Iisalmi': 'North Savo', 'Iitti': 'Päijät-Häme',
    'Ikaalinen': 'Pirkanmaa', 'Ilmajoki': 'South Ostrobothnia', 'Ilomantsi': 'North Karelia',
    'Imatra': 'South Karelia', 'Inari': 'Lapland', 'Ingå': 'Uusimaa',
    'Isojoki': 'South Ostrobothnia', 'Isokyrö': 'South Ostrobothnia', 'Jakobstad': 'Ostrobothnia',
    'Janakkala': 'Kanta-Häme', 'Joensuu': 'North Karelia', 'Jokioinen': 'Kanta-Häme',
    'Joroinen': 'North Savo', 'Joutsa': 'Central Finland', 'Juuka': 'North Karelia',
    'Juupajoki': 'Pirkanmaa', 'Juva': 'South Savo', 'Jyväskylä': 'Central Finland',
    'Jämijärvi': 'Satakunta', 'Jämsä': 'Central Finland', 'Järvenpää': 'Uusimaa',
    'Kaarina': 'Southwest Finland', 'Kaavi': 'North Savo', 'Kajaani': 'Kainuu',
    'Kalajoki': 'North Ostrobothnia', 'Kangasala': 'Pirkanmaa', 'Kangasniemi': 'South Savo',
    'Kankaanpää': 'Satakunta', 'Kannonkoski': 'Central Finland', 'Kannus': 'Central Ostrobothnia',
    'Karijoki': 'South Ostrobothnia', 'Karkkila': 'Uusimaa', 'Karstula': 'Central Finland',
    'Karvia': 'Satakunta', 'Kaskinen': 'Ostrobothnia', 'Kauhajoki': 'South Ostrobothnia',
    'Kauhava': 'South Ostrobothnia', 'Kauniainen': 'Uusimaa', 'Kaustinen': 'Central Ostrobothnia',
    'Keitele': 'North Savo', 'Kemi': 'Lapland', 'Kemijärvi': 'Lapland',
    'Keminmaa': 'Lapland', 'Kempele': 'North Ostrobothnia', 'Kerava': 'Uusimaa',
    'Keuruu': 'Central Finland', 'Kihniö': 'Pirkanmaa', 'Kimitoön': 'Southwest Finland',
    'Kinnula': 'Central Finland', 'Kirkkonummi': 'Uusimaa', 'Kitee': 'North Karelia',
    'Kittilä': 'Lapland', 'Kiuruvesi': 'North Savo', 'Kivijärvi': 'Central Finland',
    'Kokemäki': 'Satakunta', 'Kokkola': 'Central Ostrobothnia', 'Kolari': 'Lapland',
    'Konnevesi': 'Central Finland', 'Kontiolahti': 'North Karelia', 'Korsholm': 'Ostrobothnia',
    'Korsnäs': 'Ostrobothnia', 'Koski Tl': 'Southwest Finland', 'Kotka': 'Kymenlaakso',
    'Kouvola': 'Kymenlaakso', 'Kristinestad': 'Ostrobothnia', 'Kronoby': 'Ostrobothnia',
    'Kuhmo': 'Kainuu', 'Kuhmoinen': 'Pirkanmaa', 'Kuopio': 'North Savo',
    'Kuortane': 'South Ostrobothnia', 'Kurikka': 'South Ostrobothnia', 'Kustavi': 'Southwest Finland',
    'Kuusamo': 'North Ostrobothnia', 'Kyyjärvi': 'Central Finland', 'Kärkölä': 'Päijät-Häme',
    'Kärsämäki': 'North Ostrobothnia', 'Lahti': 'Päijät-Häme', 'Laihia': 'Ostrobothnia',
    'Laitila': 'Southwest Finland', 'Lapinjärvi': 'Uusimaa', 'Lapinlahti': 'North Savo',
    'Lappajärvi': 'South Ostrobothnia', 'Lappeenranta': 'South Karelia', 'Lapua': 'South Ostrobothnia',
    'Larsmo': 'Ostrobothnia', 'Laukaa': 'Central Finland', 'Lemi': 'South Karelia',
    'Lempäälä': 'Pirkanmaa', 'Leppävirta': 'North Savo', 'Lestijärvi': 'Central Ostrobothnia',
    'Lieksa': 'North Karelia', 'Lieto': 'Southwest Finland', 'Liminka': 'North Ostrobothnia',
    'Liperi': 'North Karelia', 'Lohja': 'Uusimaa', 'Loimaa': 'Southwest Finland',
    'Loppi': 'Kanta-Häme', 'Loviisa': 'Uusimaa', 'Luhanka': 'Central Finland',
    'Lumijoki': 'North Ostrobothnia', 'Luumäki': 'South Karelia', 'Malax': 'Ostrobothnia',
    'Marttila': 'Southwest Finland', 'Masku': 'Southwest Finland', 'Merijärvi': 'North Ostrobothnia',
    'Merikarvia': 'Satakunta', 'Miehikkälä': 'Kymenlaakso', 'Mikkeli': 'South Savo',
    'Muhos': 'North Ostrobothnia', 'Multia': 'Central Finland', 'Muonio': 'Lapland',
    'Muurame': 'Central Finland', 'Mynämäki': 'Southwest Finland', 'Myrskylä': 'Uusimaa',
    'Mäntsälä': 'Uusimaa', 'Mänttä-Vilppula': 'Pirkanmaa', 'Mäntyharju': 'South Savo',
    'Naantali': 'Southwest Finland', 'Nakkila': 'Satakunta', 'Nivala': 'North Ostrobothnia',
    'Nokia': 'Pirkanmaa', 'Nousiainen': 'Southwest Finland', 'Nurmes': 'North Karelia',
    'Nurmijärvi': 'Uusimaa', 'Nykarleby': 'Ostrobothnia', 'Närpes': 'Ostrobothnia',
    'Orimattila': 'Päijät-Häme', 'Oripää': 'Southwest Finland', 'Orivesi': 'Pirkanmaa',
    'Oulainen': 'North Ostrobothnia', 'Oulu': 'North Ostrobothnia', 'Outokumpu': 'North Karelia',
    'Padasjoki': 'Päijät-Häme', 'Paimio': 'Southwest Finland', 'Paltamo': 'Kainuu',
    'Pargas': 'Southwest Finland', 'Parikkala': 'South Karelia', 'Parkano': 'Pirkanmaa',
    'Pedersöre': 'Ostrobothnia', 'Pelkosenniemi': 'Lapland', 'Pello': 'Lapland',
    'Perho': 'Central Ostrobothnia', 'Petäjävesi': 'Central Finland', 'Pieksämäki': 'South Savo',
    'Pielavesi': 'North Savo', 'Pihtipudas': 'Central Finland', 'Pirkkala': 'Pirkanmaa',
    'Polvijärvi': 'North Karelia', 'Pomarkku': 'Satakunta', 'Pori': 'Satakunta',
    'Pornainen': 'Uusimaa', 'Porvoo': 'Uusimaa', 'Posio': 'Lapland',
    'Pudasjärvi': 'North Ostrobothnia', 'Pukkila': 'Uusimaa', 'Punkalaidun': 'Pirkanmaa',
    'Puolanka': 'Kainuu', 'Puumala': 'South Savo', 'Pyhtää': 'Kymenlaakso',
    'Pyhäjoki': 'North Ostrobothnia', 'Pyhäjärvi': 'North Ostrobothnia', 'Pyhäntä': 'North Ostrobothnia',
    'Pyhäranta': 'Southwest Finland', 'Pälkäne': 'Pirkanmaa', 'Pöytyä': 'Southwest Finland',
    'Raahe': 'North Ostrobothnia', 'Raasepori': 'Uusimaa', 'Raisio': 'Southwest Finland',
    'Rantasalmi': 'South Savo', 'Ranua': 'Lapland', 'Rauma': 'Satakunta',
    'Rautalampi': 'North Savo', 'Rautavaara': 'North Savo', 'Rautjärvi': 'South Karelia',
    'Reisjärvi': 'North Ostrobothnia', 'Riihimäki': 'Kanta-Häme', 'Ristijärvi': 'Kainuu',
    'Rovaniemi': 'Lapland', 'Ruokolahti': 'South Karelia', 'Ruovesi': 'Pirkanmaa',
    'Rusko': 'Southwest Finland', 'Rääkkylä': 'North Karelia', 'Saarijärvi': 'Central Finland',
    'Salla': 'Lapland', 'Salo': 'Southwest Finland', 'Sastamala': 'Pirkanmaa',
    'Sauvo': 'Southwest Finland', 'Savitaipale': 'South Karelia', 'Savonlinna': 'South Savo',
    'Savukoski': 'Lapland', 'Seinäjoki': 'South Ostrobothnia', 'Sievi': 'North Ostrobothnia',
    'Siikainen': 'Satakunta', 'Siikajoki': 'North Ostrobothnia', 'Siikalatva': 'North Ostrobothnia',
    'Siilinjärvi': 'North Savo', 'Simo': 'Lapland', 'Sipoo': 'Uusimaa',
    'Siuntio': 'Uusimaa', 'Sodankylä': 'Lapland', 'Soini': 'South Ostrobothnia',
    'Somero': 'Southwest Finland', 'Sonkajärvi': 'North Savo', 'Sotkamo': 'Kainuu',
    'Sulkava': 'South Savo', 'Suomussalmi': 'Kainuu', 'Suonenjoki': 'North Savo',
    'Sysmä': 'Päijät-Häme', 'Säkylä': 'Satakunta', 'Taipalsaari': 'South Karelia',
    'Taivalkoski': 'North Ostrobothnia', 'Taivassalo': 'Southwest Finland', 'Tammela': 'Kanta-Häme',
    'Tampere': 'Pirkanmaa', 'Tervo': 'North Savo', 'Tervola': 'Lapland',
    'Teuva': 'South Ostrobothnia', 'Tohmajärvi': 'North Karelia', 'Toholampi': 'Central Ostrobothnia',
    'Toivakka': 'Central Finland', 'Tornio': 'Lapland', 'Turku': 'Southwest Finland',
    'Tuusniemi': 'North Savo', 'Tuusula': 'Uusimaa', 'Tyrnävä': 'North Ostrobothnia',
    'Ulvila': 'Satakunta', 'Urjala': 'Pirkanmaa', 'Utajärvi': 'North Ostrobothnia',
    'Utsjoki': 'Lapland', 'Uurainen': 'Central Finland', 'Uusikaupunki': 'Southwest Finland',
    'Vaala': 'North Ostrobothnia', 'Vaasa': 'Ostrobothnia', 'Valkeakoski': 'Pirkanmaa',
    'Vantaa': 'Uusimaa', 'Varkaus': 'North Savo', 'Vehmaa': 'Southwest Finland',
    'Vesanto': 'North Savo', 'Vesilahti': 'Pirkanmaa', 'Veteli': 'Central Ostrobothnia',
    'Vieremä': 'North Savo', 'Vihti': 'Uusimaa', 'Viitasaari': 'Central Finland',
    'Vimpeli': 'South Ostrobothnia', 'Virolahti': 'Kymenlaakso', 'Virrat': 'Pirkanmaa',
    'Vöyri': 'Ostrobothnia', 'Ylitornio': 'Lapland', 'Ylivieska': 'North Ostrobothnia',
    'Ylöjärvi': 'Pirkanmaa', 'Ypäjä': 'Kanta-Häme', 'Ähtäri': 'South Ostrobothnia',
    'Äänekoski': 'Central Finland',
}

df = pd.read_csv(INPUT_DIR / "new_car_registrations_FI.csv", encoding="latin-1")

# Special areas that aren't real municipalities, but should still be kept as their own
# rows in silver, with both region and municipality set to the same label.
SPECIAL_AREAS = {"Unknown", "Foreign countries"}

# Keep rows whose Area is either a known municipality or one of the special areas
# (drops MAINLAND FINLAND and the MKxx region summary rows automatically, since
# those aren't in the map and aren't in SPECIAL_AREAS)
df_municipalities = df[
    df["Area"].isin(MUNICIPALITY_TO_REGION) | df["Area"].isin(SPECIAL_AREAS)
].copy()

# Look up region by municipality name; for special areas, region = the area name itself
df_municipalities["region"] = df_municipalities["Area"].map(MUNICIPALITY_TO_REGION)
df_municipalities.loc[df_municipalities["Area"].isin(SPECIAL_AREAS), "region"] = df_municipalities["Area"]

# Split the Month column: "2016M01" -> year=2016, month=01
df_municipalities["year"] = df_municipalities["Month"].str.split("M").str[0]
df_municipalities["month"] = df_municipalities["Month"].str.split("M").str[1]

df_municipalities["country"] = "Finland"

# Map Swedish-only municipality names to their official Finnish names.
# Source: Statistics Finland (stat.fi) municipality classification.
# Note: Korsnäs has no separate Finnish name (it's a unilingually Swedish municipality).
SWEDISH_TO_FINNISH_NAME = {
    "Ingå": "Inkoo",
    "Jakobstad": "Pietarsaari",
    "Kimitoön": "Kemiönsaari",
    "Korsholm": "Mustasaari",
    "Kristinestad": "Kristiinankaupunki",
    "Kronoby": "Kruunupyy",
    "Larsmo": "Luoto",
    "Malax": "Maalahti",
    "Nykarleby": "Uusikaarlepyy",
    "Närpes": "Närpiö",
    "Pargas": "Parainen",
    "Pedersöre": "Pedersören kunta",
}

df_municipalities["municipality"] = df_municipalities["Area"].replace(SWEDISH_TO_FINNISH_NAME)

# Map source Driving power values to standardized silver-level categories
DRIVING_POWER_MAP = {
    "Petrol": "petrol",
    "Diesel": "diesel",
    "Electricity": "electricity",
    "Natural gas (CNG)": "gas/gas flex",
    "Petrol/CNG": "gas/gas flex",
    "Petrol/Electricity (plug-in hybrid)": "plug-in hybrid",
    "Diesel/Electricity (plug-in hybrid)": "plug-in hybrid",
    "Petrol/Ethanol": "petrol/ethanol",
    "Other": "other",
}

# Drop "Total" rows entirely
df_municipalities = df_municipalities[df_municipalities["Driving power"] != "Total"].copy()

df_municipalities["driving_power"] = df_municipalities["Driving power"].map(DRIVING_POWER_MAP)

df_silver = df_municipalities.rename(columns={
    "First registrations of passenger cars": "number_of_new_registrations",
})[[
    "country",
    "year",
    "month",
    "region",
    "municipality",
    "number_of_new_registrations",
    "driving_power",
    "fetch_timestamp",
]]

# Convert types for consistency before saving (important for joining with Sweden data later)
df_silver["year"] = df_silver["year"].astype(int)
df_silver["month"] = df_silver["month"].astype(int)
df_silver["number_of_new_registrations"] = df_silver["number_of_new_registrations"].astype(int)
df_silver["fetch_timestamp"] = pd.to_datetime(df_silver["fetch_timestamp"])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_silver.to_csv(OUTPUT_DIR / "silver_car_registrations_finland.csv", index=False)